# Reconocimiento de Dígitos con Redes Neuronales Convolucionales (CNN)
---
## 1. Business Case Discovery

### 1.1 Contexto del Negocio y Antecedentes
Una importante compañía de envíos busca modernizar su sistema de ruteo de correspondencia mediante el reconocimiento automático de códigos postales. Actualmente, el proceso manual genera demoras y errores operativos. 

### 1.2 Objetivo del Proyecto
Desarrollar un sistema basado en redes neuronales convolucionales (CNN) que identifique correctamente los dígitos en imágenes de paquetes, automatizando la clasificación según el código postal.

### 1.3 Métricas de Éxito
- **Precisión (Accuracy):** Porcentaje de dígitos reconocidos correctamente.
- **Tiempo de Inferencia:** Velocidad de procesamiento por imagen.
- **Número de Parámetros:** Eficiencia y capacidad de generalización del modelo.

---

## 2. Preparación del Entorno
Importamos las librerías necesarias para el procesamiento de datos, construcción del modelo y visualización.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Configuración de visualización
%matplotlib inline
sns.set_theme(style="whitegrid")

print(f"TensorFlow Version: {tf.__version__}")

## 3. Data Processing (Procesamiento de Datos)

### 3.1 Carga del Dataset MNIST
Utilizaremos el dataset MNIST, que contiene 70,000 imágenes de dígitos manuscritos en escala de grises (28x28 píxeles).

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Imágenes de entrenamiento: {x_train_full.shape[0]}")
print(f"Imágenes de prueba: {x_test.shape[0]}")
print(f"Tamaño de imagen: {x_train_full.shape[1:]}")

### 3.2 Análisis Exploratorio y Visualización
Es crucial entender cómo se distribuyen nuestros datos para detectar posibles sesgos.

In [ ]:
# Visualizar múltiples muestras de dígitos
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train_full[i], cmap='gray')
    plt.title(f"Dígito: {y_train_full[i]}")
    plt.axis('off')
plt.suptitle("Ejemplos de Dígitos del Dataset MNIST")
plt.tight_layout()
plt.show()

# Analizar la distribución de las clases
plt.figure(figsize=(10, 4))
sns.countplot(x=y_train_full, palette="viridis")
plt.title("Distribución de Clases en el Set de Entrenamiento")
plt.xlabel("Dígito")
plt.ylabel("Frecuencia")
plt.show()

print("Distribución numérica:")
unique, counts = np.unique(y_train_full, return_counts=True)
print(dict(zip(unique, counts)))

### 3.3 Limpieza y Preprocesamiento
- **Normalización:** Escalamos los valores de los píxeles de [0, 255] a [0, 1] para mejorar la convergencia del modelo.
- **Reshape:** Las CNN esperan una entrada de 4 dimensiones (samples, height, width, channels). Añadimos el canal 1 para escala de grises.

In [ ]:
# Normalización
x_train_full = x_train_full.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Cambio de forma para la CNN (28x28x1)
x_train_full = x_train_full.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# División en Entrenamiento (70%), Validación (15%) y Prueba (15%)
# Ya tenemos x_test (10k), dividimos x_train_full para obtener validación
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full
)

print(f"Dataset dividido en:\n- Train: {x_train.shape[0]}\n- Validation: {x_val.shape[0]}\n- Test: {x_test.shape[0]}")

## 4. Model Planning (Planificación del Modelo)

### 4.1 Definición de la Arquitectura
Diseñamos una CNN con la siguiente estructura:
1. **Conv2D + ReLU:** Detector de características (bordes).
2. **MaxPooling2D:** Reducción de dimensionalidad y ruido.
3. **Conv2D + ReLU:** Detector de patrones más complejos.
4. **MaxPooling2D:** Segunda reducción.
5. **Flatten:** Conversión de mapas de características 2D a vector.
6. **Dense:** Capas de clasificación.
7. **Dropout:** Regularización para prevenir el sobreajuste (overfitting).

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    
    # Bloque 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    # Bloque 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    # Bloque 3
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Clasificador
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5), # Desactiva el 50% de neuronas aleatoriamente durante el entrenamiento
    layers.Dense(10, activation='softmax') # Probabilidades para cada dígito (0-9)
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Model Building and Training (Construcción y Entrenamiento)

Entrenamos el modelo utilizando un flujo claro, monitorizando métricas y aplicando **Early Stopping** para evitar que el modelo aprenda de memoria el set de entrenamiento.

In [ ]:
# Callback para detener el entrenamiento si la pérdida en validación deja de mejorar
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping]
)

## 6. Evaluación y Resultados

### 6.1 Análisis Cuantitativo
Visualizamos las curvas de pérdida y precisión.

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión del Modelo')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida del Modelo')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()

plt.show()

### 6.2 Matriz de Confusión
Analizamos qué dígitos se confunden más frecuentemente entre sí.

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción del Modelo')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión')
plt.show()

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

### 6.3 Análisis de Errores
Visualizamos casos donde el modelo falló para entender la variabilidad de las imágenes.

In [ ]:
error_indices = np.where(y_pred != y_test)[0]
plt.figure(figsize=(12, 6))
for i, idx in enumerate(error_indices[:10]):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f"Real: {y_test[idx]}\nPred: {y_pred[idx]}")
    plt.axis('off')
plt.suptitle("Ejemplos de Clasificación Incorrecta")
plt.tight_layout()
plt.show()

## 7. Deployment (Despliegue)

Serializamos el modelo para poder cargarlo en un entorno de producción (ej. una API con FastAPI o una interfaz con Streamlit).

In [ ]:
model_name = 'modelo_digitos_cnn_final.keras'
model.save(model_name)
print(f"Modelo guardado exitosamente como {model_name}")